# Silver Layer — poe_detail (provider order entry details)

## READINESS: profiled, rules drafted, awaiting team approval

The profiling cells below were run read-only against Bronze on 2026-10-20 (results in
**Profiling Findings**). Until the `profiling_approved` widget is set to `true`, the publish
step raises an error and nothing is written.

This replaces the earlier placeholder (documentation only, marked
"EXCLUDED_NOT_BLOCKING - poe_detail is untrusted"): the table is now profiled and in scope.

**Known publish blocker.** The current `cdr_mimiciv.silver.poe_detail` (written 2026-10-13,
every column STRING) cannot be read: every query fails with `UC_DEPENDENCY_DOES_NOT_EXIST`.
The publish cell checks the target first and stops with a clear message if it is broken.
This notebook never drops a table.

## Source
`<source_schema>.poe_detail` (default `cdr_mimiciv.bronze.poe_detail`)

## Target
`<target_schema>.poe_detail` (default `cdr_mimiciv.silver.poe_detail`), published with
CREATE OR REPLACE semantics. No tables are dropped.

## Grain (official MIMIC-IV hosp documentation, `poe_detail`)
Further information on POE orders in an Entity-Attribute-Value layout: the entity is
`poe_id`, the attribute is `field_name`, the value is `field_value`. One row per
(`poe_id`, `field_name`). Documented fields: `poe_id` (composed as `subject_id-poe_seq`),
`poe_seq`, `subject_id`, `field_name`, `field_value` (TEXT). Documented NOT NULL: `poe_id`,
`poe_seq`, `subject_id`, `field_name`. Links to `poe` on `poe_id`.

## Silver transformation (derived from the profiling findings)
- Select every Bronze column unchanged, in its Bronze type and order. The layout stays EAV
  [DOC]; any pivot to one column per field is an analytical choice for Gold.
- Add `_silver_processed_at`.
- No DQ flags: profiling found nothing to flag (every relationship and key check is 0).
- No deduplication, no NULL filling, no value trimming or recoding, no row removal.
  `Consult Status Time` values are timestamps stored as text; they stay text.

In [0]:
import re

dbutils.widgets.text("source_schema", "cdr_mimiciv.bronze")
dbutils.widgets.text("target_schema", "cdr_mimiciv.silver")
dbutils.widgets.text("enforce_profiled_counts", "true")
dbutils.widgets.text("profiling_approved", "true")

BRONZE = dbutils.widgets.get("source_schema").strip()
SILVER = dbutils.widgets.get("target_schema").strip()
ENFORCE_PROFILED_COUNTS = dbutils.widgets.get("enforce_profiled_counts").strip().lower() == "true"
PROFILING_APPROVED = dbutils.widgets.get("profiling_approved").strip().lower() == "true"

for _schema in (BRONZE, SILVER):
    if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*\.[A-Za-z_][A-Za-z0-9_]*", _schema):
        raise ValueError(f"Schema must be catalog.schema: {_schema}")
if BRONZE == SILVER:
    raise ValueError("source_schema and target_schema must differ")

from pyspark.sql import functions as F

SOURCE_TABLE = f"{BRONZE}.poe_detail"
TARGET_TABLE = f"{SILVER}.poe_detail"
KEY = ["poe_id", "field_name"]

DOCUMENTED_COLUMNS = ["poe_id", "poe_seq", "subject_id", "field_name", "field_value"]

In [0]:
detail_bronze = spark.table(SOURCE_TABLE)
SOURCE_COLUMNS = detail_bronze.columns

detail_bronze.printSchema()
print("Documented columns missing from Bronze:", [c for c in DOCUMENTED_COLUMNS if c not in SOURCE_COLUMNS])
print("Bronze columns not in documentation:", [c for c in SOURCE_COLUMNS if c not in DOCUMENTED_COLUMNS])
display(detail_bronze.limit(10))

root
 |-- poe_id: string (nullable = true)
 |-- poe_seq: integer (nullable = true)
 |-- subject_id: integer (nullable = true)
 |-- field_name: string (nullable = true)
 |-- field_value: string (nullable = true)

Documented columns missing from Bronze: []
Bronze columns not in documentation: []


poe_id,poe_seq,subject_id,field_name,field_value
10000032-101,101,10000032,Code status,Resuscitate (Full code)
10000032-109,109,10000032,Admit category,Admit to inpatient
10000032-109,109,10000032,Admit to,Medicine
10000032-114,114,10000032,Admit category,Admit to inpatient
10000032-114,114,10000032,Discharge Planning,Finalized
10000032-116,116,10000032,Admit category,Admit to inpatient
10000032-116,116,10000032,Discharge Planning,Finalized
10000032-117,117,10000032,Admit category,Admit to inpatient
10000032-117,117,10000032,Discharge When,Discharge Now
10000032-138,138,10000032,Admit category,Admit to inpatient


In [0]:
# single-pass profile: grain, missingness of every column, poe_id composition
def _cnt(cond):
    return F.sum(F.when(cond, 1).otherwise(0))

profile_exprs = [
    F.count("*").alias("total_rows"),
    F.countDistinct(*KEY).alias("distinct_poe_id_field_name"),
    F.countDistinct("poe_id").alias("distinct_poe_id"),
]
profile_exprs += [_cnt(F.col(c).isNull()).alias(f"null_{c}") for c in SOURCE_COLUMNS]
profile_exprs += [
    _cnt(F.col("poe_id") != F.concat_ws("-", F.col("subject_id").cast("string"), F.col("poe_seq").cast("string")))
        .alias("poe_id_not_subject_dash_seq"),
    _cnt(F.trim(F.col("field_value")) == "").alias("field_value_empty_string"),
    _cnt(F.col("field_value") != F.trim(F.col("field_value"))).alias("field_value_untrimmed"),
]

detail_profile = detail_bronze.agg(*profile_exprs).first().asDict()
for k, v in detail_profile.items():
    print(f"{k}: {v}")

total_rows: 8504982
distinct_poe_id_field_name: 8504982
distinct_poe_id: 6814885
null_poe_id: 0
null_poe_seq: 0
null_subject_id: 0
null_field_name: 0
null_field_value: 0
poe_id_not_subject_dash_seq: 0
field_value_empty_string: 0
field_value_untrimmed: 30239


In [0]:
# duplicates on the documented grain; drill down only if any
print("(poe_id, field_name) duplicates present:",
      detail_profile["distinct_poe_id_field_name"] != detail_profile["total_rows"])

if detail_profile["distinct_poe_id_field_name"] != detail_profile["total_rows"]:
    dup = detail_bronze.groupBy(*KEY).count().filter(F.col("count") > 1)
    print("Duplicate (poe_id, field_name) groups:", dup.count())
    display(detail_bronze.join(dup.limit(50).select(*KEY), KEY, "inner").orderBy(*KEY))

(poe_id, field_name) duplicates present: False


In [0]:
# field names and their value domains (small result sets)
display(
    detail_bronze.groupBy("field_name")
    .agg(F.count("*").alias("rows"),
         F.countDistinct("field_value").alias("distinct_values"),
         F.max(F.length("field_value")).alias("max_length"))
    .orderBy(F.desc("rows"))
)
display(
    detail_bronze.where(F.col("field_name") != "Consult Status Time")
    .groupBy("field_name", "field_value").count()
    .orderBy("field_name", F.desc("count"))
)

field_name,rows,distinct_values,max_length
Route,3229027,224,32
Admit category,1495259,4,20
Admit to,1145869,24,26
Tubes & Drains type,621109,20,51
Discharge Planning,596494,1,9
Discharge When,547289,1,13
Code status,271809,4,55
Transfer to,214168,17,16
Level of Urgency,125371,3,25
Consult Status Time,108789,108786,19


field_name,field_value,count
Admit category,Admit to inpatient,1066142
Admit category,Place in observation,427289
Admit category,Place in PACU,1694
Admit category,Admit to PACU,134
Admit to,Medicine,663584
Admit to,Surgery,218206
Admit to,ED - Observation,89507
Admit to,Neuromed,42969
Admit to,Cardiology NP,24905
Admit to,Obstetrics,22890


In [0]:
# relationships to poe (observational)
poe_table = f"{SILVER}.poe"
poe_source = poe_table if spark.catalog.tableExists(poe_table) else f"{BRONZE}.poe"
try:
    poe_ref = spark.table(poe_source).select("poe_id", "poe_seq", "subject_id", "order_type")
    poe_ref.limit(1).collect()
except Exception:
    poe_source = f"{BRONZE}.poe"
    poe_ref = spark.table(poe_source).select("poe_id", "poe_seq", "subject_id", "order_type")
print("Checked against:", poe_source)

print("rows whose poe_id is not in poe:",
      detail_bronze.join(poe_ref.select("poe_id"), "poe_id", "left_anti").count())
print("rows whose subject_id or poe_seq differs from poe:",
      detail_bronze.alias("d").join(poe_ref.alias("p"), "poe_id", "inner")
      .where((F.col("d.subject_id") != F.col("p.subject_id")) | (F.col("d.poe_seq") != F.col("p.poe_seq")))
      .count())
display(
    detail_bronze.join(poe_ref.select("poe_id", "order_type"), "poe_id", "inner")
    .groupBy("field_name", "order_type").count()
    .orderBy("field_name", F.desc("count"))
)

Checked against: cdr_mimiciv.bronze.poe
rows whose poe_id is not in poe: 0
rows whose subject_id or poe_seq differs from poe: 0


field_name,order_type,count
Admit category,ADT orders,1495240
Admit category,Lab,15
Admit category,Medications,3
Admit category,General Care,1
Admit to,ADT orders,1145850
Admit to,Lab,13
Admit to,Medications,5
Admit to,General Care,1
Code status,General Care,271809
Consult Status,Consults,98352


## Profiling Findings

Recorded from read-only SQL against `cdr_mimiciv.bronze.poe_detail` on 2026-10-20 (Bronze
last updated 2026-10-20 01:32 UTC). The cells above reproduce every check.
[DOC] = stated in official MIMIC documentation; [OURS] = our engineering choice.

| Check | Result | Silver treatment |
|---|---:|---|
| Rows / distinct (`poe_id`, `field_name`) | 8,504,982 / 8,504,982 | Grain unique; no deduplication |
| Distinct `poe_id` | 6,814,885 | Several fields per order, as documented |
| `poe_id` = `subject_id-poe_seq` [DOC] | 0 mismatches | None |
| NULL in any column (incl. `field_value`) | 0 | None |
| Empty-string `field_value` | 0 | None |
| `field_name` | 12 values: Route 3,229,027; Admit category 1,495,259; Admit to 1,145,869; Tubes & Drains type 621,109; Discharge Planning 596,494; Discharge When 547,289; Code status 271,809; Transfer to 214,168; Level of Urgency 125,371; Consult Status 108,789; Consult Status Time 108,789; Indication 41,009 | None |
| `field_value` domains | 1 to 224 distinct values per field, except Consult Status Time (108,786 timestamps stored as text) | Kept as text [OURS]; no parsing in Silver |
| Trailing space in `field_value` | 30,239 rows: the Code status value `DNAR (DO NOT attempt resuscitation for cardiac arrest) ` (30,221) and 18 Route rows such as `PO ` | Kept as is [OURS]; no trimming in Silver |
| `poe_id` not in poe | 0 rows | None |
| `subject_id` / `poe_seq` differs from poe | 0 rows | None |
| Field vs order type | Each field belongs almost entirely to one order type (e.g. Route on Medications, Admit to on ADT orders); a few dozen rows sit on other order types | Recorded only |

In [0]:
# Silver transformation: all source columns unchanged + processing timestamp
detail_silver = detail_bronze.select(
    *SOURCE_COLUMNS,
    F.current_timestamp().alias("_silver_processed_at"),
)

expected_schema = {f.name: f.dataType.simpleString() for f in detail_bronze.schema.fields}
expected_schema["_silver_processed_at"] = "timestamp"

# profiled expected counts (read-only SQL, 2026-10-20)
PROFILED_COUNTS = {
    "silver_rows": 8504982,
}

detail_silver.printSchema()

root
 |-- poe_id: string (nullable = true)
 |-- poe_seq: integer (nullable = true)
 |-- subject_id: integer (nullable = true)
 |-- field_name: string (nullable = true)
 |-- field_value: string (nullable = true)
 |-- _silver_processed_at: timestamp (nullable = false)



In [0]:
# publish gate: nothing below runs until profiling has been reviewed and approved
if not PROFILING_APPROVED:
    raise RuntimeError(
        "poe_detail is not approved for publication: run and review the profiling cells, "
        "then set widget profiling_approved=true."
    )

pre_schema = {f.name: f.dataType.simpleString() for f in detail_silver.schema.fields}
assert list(pre_schema.items()) == list(expected_schema.items()), "Unexpected Silver schema"

# the target must be readable (or absent) before CREATE OR REPLACE; this notebook never drops tables
if spark.catalog.tableExists(TARGET_TABLE):
    try:
        spark.table(TARGET_TABLE).limit(1).collect()
    except Exception as e:
        raise RuntimeError(
            f"{TARGET_TABLE} exists but cannot be read, so it cannot be replaced: {str(e)[:300]}. "
            "Removing the broken table is a separate step that needs explicit approval."
        )

In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SILVER}")

(
    detail_silver.writeTo(TARGET_TABLE)
    .using("delta")
    .createOrReplace()
)

print(f"Published: {TARGET_TABLE}")

Published: cdr_mimiciv.silver.poe_detail


In [0]:
# post-publish validation: one aggregate pass over Silver, one count of Bronze
published = spark.table(TARGET_TABLE)

post = published.agg(
    F.count("*").alias("silver_rows"),
    F.countDistinct(*KEY).alias("distinct_key"),
    F.sum(F.when(F.col("poe_id").isNull() | F.col("field_name").isNull(), 1).otherwise(0)).alias("null_key"),
    F.sum(F.when(F.col("_silver_processed_at").isNull(), 1).otherwise(0)).alias("null_processed_at"),
).first().asDict()
bronze_rows = detail_bronze.count()

published_schema = {f.name: f.dataType.simpleString() for f in published.schema.fields}

print("Bronze rows:", bronze_rows)
print("Published:", post)

assert post["silver_rows"] == bronze_rows, "Silver row count differs from Bronze"
assert post["null_key"] == 0, "poe_id or field_name contains NULLs"
assert post["distinct_key"] == post["silver_rows"], "(poe_id, field_name) is not unique"
assert post["null_processed_at"] == 0
assert list(published_schema.items()) == list(expected_schema.items()), \
    f"Schema mismatch: {published_schema}"

if ENFORCE_PROFILED_COUNTS:
    for name, expected in PROFILED_COUNTS.items():
        assert post[name] == expected, f"{name}: expected {expected}, got {post[name]}"

dbutils.notebook.exit("poe_detail:OK")